In [32]:
# Group Project: Depression, Comorbidity, and Multiple Linear Regression
# by Raquel Gonzalez, Nancy Hernandez, Grant Pasamonte, and Kevin Roldan
# 2026-10-26
# Load necessary libraries
import pandas as pd
import numpy as np
from pyprojroot import here # Mimics R's here() function to find project root

In [33]:
# Question 2. Data audit, cleaning, and analytic cohort
# 2a. Standardization
# Use pd.read_csv to import CSV, use here() to pull direct file path, set dtype = str as per Q2b instruction 
raw_data = pd.read_csv(here("data/PUBH551_Group_Project_Depression_Data_2005_2025.csv"), dtype = str)

In [34]:
# Identify the 5-missing value representations in raw data by quickly looking at every unique response
# Do not include the variables containing "id", "date", and "USD" as those will have an abundance of unique values and will be hard to sift through
unique = raw_data.drop(
    columns = [
        col
       # Loop through each column and only include those not containing "_id", "date", "usd"
        for col in raw_data.columns
        if any(x in col.lower() for x in ["_id", "date", "usd"])
    ]
)
# Loop through each column in "unique" and show the top 10 most common text values to identify the missing value representations
for col in unique.columns:
    print(unique[col].value_counts().head(50))
# 5 missing value representations = "", ".", "NA", "N/A" , and "missing"

year
2005    348
2006    348
2017    344
2013    343
2025    343
2023    343
2015    343
2010    342
2019    342
2018    342
2020    342
2008    342
2024    342
2016    342
2014    342
2009    342
2021    342
2007    342
2011    342
2012    342
2022    342
Name: count, dtype: int64
country
United States    7200
Name: count, dtype: int64
comorbid_condition
Asthma                      999
Obesity                     999
Hypertension                999
Chronic Kidney Disease      943
Cardiovascular Disease      943
Type 2 Diabetes             943
OBESITY                      93
CHRONIC KIDNEY DISEASE       93
CARDIOVASCULAR DISEASE       93
ASTHMA                       93
TYPE 2 DIABETES              93
HYPERTENSION                 93
chronic kidney disease       65
type 2 diabetes              65
hypertension                 65
asthma                       65
obesity                      65
cardiovascular disease       65
Type_2_Diabetes              56
Cardiovascular_Disease       56
Ch

In [35]:
# Use replace() to replace the missing value representations with standard NaN values across all columns
na_data = raw_data.replace(["", ".", "NA", "N/A", "missing"], np.nan)
# Standardize the dataset to fix inconsistent capitalization, whitespace, underscores, and missing-value representations
data = na_data.copy()
# Exclude the often case-sensitive ID codes from cleaning
exclude_cols = ["record_id", "participant_id"]
# For every column in data (not excluded above), 
for col in data:
    if col not in exclude_cols:
        data[col] = (
            data[col]
            .astype(str)
            # Convert every letter to lowercase
            .str.lower()
            # Removes leading/trailing spaces    
            .str.strip()
            # Rreplace all underscores with normal spaces
            .str.replace("_", " ", regex = False)
        )

In [22]:
# 2b. Cleaning Numeric Fields
# Retroactively add dtype = str to the CSV import step to have deliberate control of column type
# Define numeric fields containing commas, percent signs, currency symbols, units, and other extra characters
numeric_cols = [
    "annual_household_income_usd",
    "area_deprivation_index_percentile",
    "sleep_duration_hours",
    "physical_activity_min_week",
    "bmi_kg_m2",
    "systolic_bp_mmHg",
    "total_cholesterol_mg_dL",
    "healthcare_visits_past_year",
    "phq9_score"
]
# Use pandas to parse numbers similar to parse_number() R function
for col in numeric_cols:
    data[col] = pd.to_numeric(
        # Use regex to strip out letters and symbols, then force to numeric using pd.to_numeric() above
        data[col].astype(str).str.replace(r"[^\d.-]", "", regex = True),
        errors = "coerce"
    )

In [36]:
# 2c. Removing Duplicates
# Use drop_duplicates() to remove all duplicate rows
data = data.drop_duplicates()
# Identify rows where either prevalence or mortality values are missing
missing_vars = data[
    data["condition_prevalence_per_100k"].isna() |
    data["condition_mortality_per_100k"].isna()
    ][[
     # Select key columns (record_id), comorbid_condition) in addition to prevalence/mortality to create a df for easy scanning
    "record_id",
    "comorbid_condition",
    "condition_prevalence_per_100k",
    "condition_mortality_per_100k"
    ]]
# Print table of missing variables with key identifier columns
print(missing_vars)

     record_id      comorbid_condition condition_prevalence_per_100k  \
175    R500308  cardiovascular disease                        9991.0   
1115   R600308                 obesity                       31353.0   
1755   R100102            hypertension                           NaN   
1843   R300308                  asthma                        8328.1   
2837   R200102         type 2 diabetes                           NaN   
4545   R100308            hypertension                       27786.9   
6224   R200308         type 2 diabetes                       12180.0   
6451   R600102                 obesity                           NaN   
6520   R300102                  asthma                           NaN   
6573   R500102  cardiovascular disease                           NaN   
6876   R400102  chronic kidney disease                           NaN   
7199   R400308  chronic kidney disease                       15360.7   

     condition_mortality_per_100k  
175                        

In [37]:
# 2d. Suspicious Value
# Use describe() to identify the suspicious value in total_cholesterol_mg_dL
print(data["total_cholesterol_mg_dL"].describe())
# 950 is extreme outlier

count     6954
unique     459
top        174
freq        80
Name: total_cholesterol_mg_dL, dtype: object


In [38]:
# 2e. Complete-Case Analysis for Primary MLR
# Perform a complete-case analysis for the primary MLR
# Filter the dataset for our group's chosen condition subset
primary_vars = [
    "phq9_score",
    "physical_activity_min_week",
    "age_years",
    "sex",
    "annual_household_income_usd",
    "smoking_status"
]
analytic_data = (
    data[data["comorbid_condition"] == "cardiovascular disease"]
    .dropna(subset = primary_vars)
)
# Export the cleaned analytic CSV for analysis into the data folder
analytic_data.to_csv(here("data/cleaned_analytic_data.csv"), index = False)